# 06a — Model Deployment

A model that only runs in a notebook is useless. Deployment makes it accessible to users, front-end apps, mobile clients, and other services. This notebook covers the full path from a trained model object to a production-ready API running inside a Docker container.

---
## 1 · Saving & Loading Models

Before you can serve a model, you need to **serialize** it — convert the in-memory object to a file on disk. Different frameworks have different conventions.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print(f"Test accuracy: {clf.score(X_test, y_test):.3f}")

### pickle — Python's built-in serializer

Works for any Python object. Fast and simple, but files are Python-version-sensitive and **not secure** — never unpickle data from untrusted sources.

In [ ]:
import pickle

with open("model_pickle.pkl", "wb") as f:
    pickle.dump(clf, f)

with open("model_pickle.pkl", "rb") as f:
    loaded = pickle.load(f)

loaded.predict(X_test[:3])

### joblib — optimized for NumPy-heavy objects

scikit-learn's recommended approach. Handles large NumPy arrays more efficiently than pickle and supports compression out of the box.

In [ ]:
import joblib

joblib.dump(clf, "model_joblib.pkl")

loaded = joblib.load("model_joblib.pkl")
loaded.predict(X_test[:3])

### torch.save — PyTorch models

PyTorch models are saved as `state_dict` (just the weights) or as the full model object. Saving `state_dict` is preferred because it decouples the architecture from the weights.

In [ ]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 3)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

net = SimpleNet()

torch.save(net.state_dict(), "model_torch.pt")

loaded_net = SimpleNet()
loaded_net.load_state_dict(torch.load("model_torch.pt", weights_only=True))
loaded_net.eval()

sample = torch.tensor(X_test[:3], dtype=torch.float32)
with torch.no_grad():
    print(loaded_net(sample))

### ONNX — framework-agnostic format

ONNX (Open Neural Network Exchange) lets you export a model from one framework and run it in another. Train in PyTorch, serve with ONNX Runtime in C++ — useful when you need maximum inference speed or cross-platform compatibility.

In [ ]:
dummy_input = torch.randn(1, 4)

torch.onnx.export(
    net,
    dummy_input,
    "model.onnx",
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={"features": {0: "batch"}, "logits": {0: "batch"}},
)
print("Exported to ONNX")

import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("model.onnx")
result = session.run(None, {"features": X_test[:3].astype(np.float32)})
print("ONNX predictions:", result[0])

---
## 2 · Serving Models with FastAPI

FastAPI is a modern Python web framework that's become the go-to for ML serving. It's fast (async, built on Starlette), generates automatic API docs, and uses Pydantic for input validation — catching bad inputs before they reach your model.

### Step 1 — Train and save a model

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)
joblib.dump(model, "iris_model.pkl")
print("Model saved ✓")

### Step 2 — Build the FastAPI app

This is the complete `app.py` file. It loads the model once at startup, validates every incoming request with Pydantic, and returns structured JSON predictions.

In [ ]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import numpy as np

app = FastAPI(title="Iris Classifier API")

model = joblib.load("iris_model.pkl")
CLASS_NAMES = ["setosa", "versicolor", "virginica"]


class IrisFeatures(BaseModel):
    sepal_length: float = Field(..., gt=0, description="cm")
    sepal_width: float  = Field(..., gt=0, description="cm")
    petal_length: float = Field(..., gt=0, description="cm")
    petal_width: float  = Field(..., gt=0, description="cm")


class Prediction(BaseModel):
    predicted_class: str
    confidence: float
    probabilities: dict[str, float]


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict", response_model=Prediction)
def predict(features: IrisFeatures):
    X = np.array([[features.sepal_length, features.sepal_width,
                   features.petal_length, features.petal_width]])
    proba = model.predict_proba(X)[0]
    idx = int(np.argmax(proba))
    return Prediction(
        predicted_class=CLASS_NAMES[idx],
        confidence=round(float(proba[idx]), 4),
        probabilities={name: round(float(p), 4) for name, p in zip(CLASS_NAMES, proba)},
    )

### Step 3 — Run and test the API

Start the server with `uvicorn app:app --reload`, then test it. The automatic docs live at `http://127.0.0.1:8000/docs`.

In [ ]:
import requests

payload = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2,
}

response = requests.post("http://127.0.0.1:8000/predict", json=payload)
print(response.json())

Equivalent curl command:
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}'
```

---
## 3 · Input Validation with Pydantic

Without validation, bad inputs silently produce garbage predictions. Pydantic catches errors at the API boundary — before data reaches your model.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class HousePriceInput(BaseModel):
    sqft: float = Field(..., gt=0, lt=100_000)
    bedrooms: int = Field(..., ge=0, le=20)
    zip_code: str = Field(..., pattern=r"^\d{5}$")
    property_type: Literal["house", "condo", "townhouse"]

    @field_validator("sqft")
    @classmethod
    def sqft_reasonable(cls, v):
        if v < 100:
            raise ValueError("sqft below 100 is likely an error")
        return v

valid = HousePriceInput(sqft=1500, bedrooms=3, zip_code="90210", property_type="house")
print(valid.model_dump())

try:
    HousePriceInput(sqft=-1, bedrooms=3, zip_code="abc", property_type="castle")
except Exception as e:
    print(f"\nValidation errors:\n{e}")

---
## 4 · Batch Prediction vs Real-Time

| | **Real-Time** | **Batch** |
|---|---|---|
| **How** | API responds per-request | Script processes a file overnight |
| **Latency** | Milliseconds | Minutes to hours |
| **When** | User-facing (search, recommendations) | Reporting, scoring a whole customer list |
| **Infra** | Always-on server (FastAPI, Flask) | Cron job, Airflow, Spark |
| **Cost** | Higher (server always running) | Lower (spin up, process, shut down) |

In [ ]:
import pandas as pd
import joblib

model = joblib.load("iris_model.pkl")

batch_data = pd.DataFrame({
    "sepal_length": [5.1, 6.2, 4.9, 7.0],
    "sepal_width":  [3.5, 2.8, 3.1, 3.2],
    "petal_length": [1.4, 4.8, 1.5, 4.7],
    "petal_width":  [0.2, 1.8, 0.1, 1.4],
})

batch_data["prediction"] = model.predict(batch_data.values)
batch_data["confidence"] = model.predict_proba(batch_data.values).max(axis=1).round(3)

batch_data.to_csv("predictions_output.csv", index=False)
batch_data

---
## 5 · Docker for ML

"It works on my machine" is the most expensive sentence in software. Docker packages your code, dependencies, and model into a **container** that runs identically everywhere — your laptop, a colleague's machine, or a cloud server.

**Why Docker for ML specifically?**
- ML has notoriously fragile dependency chains (numpy, scipy, torch versions all matter)
- Models need the exact same library versions at training and serving time
- Containers make deployment reproducible and scalable

### Dockerfile for our FastAPI ML API

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY iris_model.pkl .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile requirements.txt
fastapi>=0.100
uvicorn[standard]>=0.23
scikit-learn>=1.3
joblib>=1.3
numpy>=1.24

### Build and run

```bash
docker build -t iris-api .
docker run -p 8000:8000 iris-api
```

The API is now running inside a container. Same curl command works:

```bash
curl http://localhost:8000/health
```

### Key Dockerfile instructions

| Instruction | Purpose |
|---|---|
| `FROM` | Base image — start from an official Python image |
| `WORKDIR` | Set the working directory inside the container |
| `COPY` | Copy files from your machine into the container |
| `RUN` | Execute a command during build (install deps) |
| `EXPOSE` | Document which port the app listens on |
| `CMD` | Default command when the container starts |

---
## 6 · Docker Compose — Multi-Container Apps

Real deployments rarely have just one service. You might need an API + a database + a cache. Docker Compose orchestrates multiple containers with a single YAML file.

In [ ]:
%%writefile docker-compose.yml
services:
  api:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - db
    environment:
      - DATABASE_URL=postgresql://user:pass@db:5432/mldb

  db:
    image: postgres:15
    environment:
      POSTGRES_USER: user
      POSTGRES_PASSWORD: pass
      POSTGRES_DB: mldb
    volumes:
      - pgdata:/var/lib/postgresql/data

volumes:
  pgdata:

```bash
docker compose up --build      # start everything
docker compose down             # tear it all down
docker compose logs api         # check API logs
```

---
## 7 · Cloud Deployment Overview

Once your container works locally, you need somewhere to run it in the cloud. Here's the landscape:

| Provider | Managed ML | Container-based | Serverless |
|---|---|---|---|
| **AWS** | SageMaker | ECS / EKS | Lambda |
| **GCP** | Vertex AI | Cloud Run / GKE | Cloud Functions |
| **Azure** | Azure ML | ACI / AKS | Azure Functions |

**When to use what:**

- **Managed ML platforms** (SageMaker, Vertex AI): full lifecycle — training, hosting, monitoring. Higher cost, less control, fastest to production.
- **Container services** (ECS, Cloud Run): deploy your Docker image directly. You control the code; they handle scaling. Best balance of control and convenience.
- **Serverless** (Lambda, Cloud Functions): for lightweight models with sporadic traffic. No server to manage, pay per invocation. Cold starts can be an issue.
- **Kubernetes** (EKS, GKE, AKS): full control, maximum complexity. Only worth it at scale or when you already have a K8s team.

In [ ]:
gcloud_deploy = """
# Deploy to Google Cloud Run (one command after pushing image)

gcloud builds submit --tag gcr.io/PROJECT_ID/iris-api

gcloud run deploy iris-api \\
    --image gcr.io/PROJECT_ID/iris-api \\
    --port 8000 \\
    --memory 1Gi \\
    --allow-unauthenticated
"""
print(gcloud_deploy.strip())

---
## Summary

| Step | Tool | Key Idea |
|---|---|---|
| Save model | joblib / torch.save / ONNX | Serialize weights to disk |
| Build API | FastAPI + Pydantic | Type-safe HTTP endpoint |
| Containerize | Docker | Reproducible environment |
| Orchestrate | Docker Compose | Multi-service setup |
| Deploy | Cloud Run / ECS / SageMaker | Scalable hosting |

The pipeline is always: **train → save → wrap in API → containerize → deploy**. Master each step and you can ship any model.